# PySpark Essentials: Practice Lesson (Toy Data)

## Purpose

This is a low-stakes practice ground before you touch real FHIR data. The toy dataset (`toy_store_data.json`) mimics the same structural challenges FHIR gave you — nested objects, arrays, cross-resource references, mixed record types in one file — but in a simple, familiar domain (customers and orders), so you can focus entirely on PySpark mechanics without also parsing unfamiliar healthcare concepts.

**What you'll practice:** reading JSON, inspecting schema, filtering, selecting/flattening nested fields, `explode`, joins across "resource types," lazy evaluation with `.explain()`, and writing a Delta table.

### Import the Toy Data

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS learning;
CREATE SCHEMA IF NOT EXISTS learning.practice;
CREATE VOLUME IF NOT EXISTS learning.practice.raw_data;

Upload `toy_store_data.json` to `learning.practice.raw_data`

In [0]:
display(dbutils.fs.ls("/Volumes/learning/practice/raw_data/"))

path,name,size,modificationTime
dbfs:/Volumes/learning/practice/raw_data/toy_store_data.json,toy_store_data.json,2157,1783841193000


Note on file format: unlike Synthea's FHIR Bundles (which are one big multi-line JSON object per file, requiring `.option("multiline", "true")`), this toy file is newline-delimited JSON — one flat JSON object per line. This is actually Spark's native/default JSON format, so you'll read it without the `multiline` option. Worth noticing the difference, since you'll need to switch that option back on when you return to FHIR files.

### Read and Inspect

In [0]:
df = spark.read.json("/Volumes/learning/practice/raw_data/toy_store_data.json")

df.printSchema()
df.show(truncate=False)

root
 |-- contacts: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- value: string (nullable = true)
 |-- customerRef: string (nullable = true)
 |-- id: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- productName: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- unitPrice: double (nullable = true)
 |-- loyaltyTier: string (nullable = true)
 |-- name: struct (nullable = true)
 |    |-- first: string (nullable = true)
 |    |-- last: string (nullable = true)
 |-- orderDate: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- signupDate: string (nullable = true)
 |-- status: string (nullable = true)

+-----------------------------------------------+-----------------+--------+-----------------------------------------------------

Notice the file mixes two record types (Customer and Order) in one read, just like a FHIR Bundle mixes resource types — Spark infers one combined schema across all of them, so you'll see columns like contacts, items, customerRef all present, even though a given row only populates the ones relevant to its resourceType.

In [0]:
df.explain()

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonJsonScan json [contacts#11700,customerRef#11701,id#11702,items#11703,loyaltyTier#11704,name#11705,orderDate#11706,resourceType#11707,signupDate#11708,status#11709] Batched: true, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/learning/practice/raw_data/toy_store_data.json], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<contacts:array<struct<type:string,value:string>>,customerRef:string,id:string,items:array<...


== Photon Explanation ==
The query is fully supported by Photon.


### Filter by Record Type

In [0]:
from pyspark.sql.functions import col

customers_raw = df.filter(col("resourceType") == "Customer")
orders_raw = df.filter(col("resourceType") == "Order")

customers_raw.show(truncate=False)
orders_raw.show(truncate=False)

+-----------------------------------------------+-----------+--------+-----+-----------+---------------+---------+------------+----------+------+
|contacts                                       |customerRef|id      |items|loyaltyTier|name           |orderDate|resourceType|signupDate|status|
+-----------------------------------------------+-----------+--------+-----+-----------+---------------+---------+------------+----------+------+
|[{email, elena@example.com}, {phone, 555-0101}]|NULL       |cust-001|NULL |gold       |{Elena, Rivera}|NULL     |Customer    |2023-01-15|NULL  |
|[{email, marcus@example.com}]                  |NULL       |cust-002|NULL |silver     |{Marcus, Chen} |NULL     |Customer    |2023-06-02|NULL  |
|[{email, priya@example.com}, {phone, 555-0103}]|NULL       |cust-003|NULL |gold       |{Priya, Patel} |NULL     |Customer    |2022-11-20|NULL  |
+-----------------------------------------------+-----------+--------+-----+-----------+---------------+---------+----------

This is the direct equivalent of filtering FHIR bundle entries by `resourceType` — same pattern, simpler data.

### Flatten Nested Fields (Customers)

In [0]:
customers_clean = customers_raw.select(
    col("id").alias("customer_id"),
    col("name.first").alias("first_name"),
    col("name.last").alias("last_name"),
    col("loyaltyTier"),
    col("signupDate").cast("date").alias("signup_date")
)

customers_clean.show()

+-----------+----------+---------+-----------+-----------+
|customer_id|first_name|last_name|loyaltyTier|signup_date|
+-----------+----------+---------+-----------+-----------+
|   cust-001|     Elena|   Rivera|       gold| 2023-01-15|
|   cust-002|    Marcus|     Chen|     silver| 2023-06-02|
|   cust-003|     Priya|    Patel|       gold| 2022-11-20|
+-----------+----------+---------+-----------+-----------+



### Explode an Array (Contacts)

In [0]:
from pyspark.sql.functions import explode

contacts_clean = customers_raw.select(
    col("id").alias("customer_id"),
    explode(col("contacts")).alias("contact")
).select(
    "customer_id",
    col("contact.type").alias("contact_type"),
    col("contact.value").alias("contact_value")
)

contacts_clean.show()

+-----------+------------+------------------+
|customer_id|contact_type|     contact_value|
+-----------+------------+------------------+
|   cust-001|       email| elena@example.com|
|   cust-001|       phone|          555-0101|
|   cust-002|       email|marcus@example.com|
|   cust-003|       email| priya@example.com|
|   cust-003|       phone|          555-0103|
+-----------+------------+------------------+



Notice `explode` turns one row (one customer with 2 contacts) into two rows (one per contact), each still tagged with `customer_id`. This is exactly the mechanic you'll use on a FHIR Bundle's `entry` array, and again inside `items` next.

### Explode a Nested Array of Objects (Order Items)
This is the closest analogue to Observation's mixed-type-value problem you'll see in FHIR — here it's simpler (every item has the same fields), but the explode-then-flatten pattern is identical:

In [0]:
order_items_clean = orders_raw.select(
    col("id").alias("order_id"),
    col("customerRef"),
    explode(col("items")).alias("item")
).select(
    "order_id",
    col("customerRef"),
    col("item.sku").alias("sku"),
    col("item.productName").alias("product_name"),
    col("item.quantity").alias("quantity"),
    col("item.unitPrice").alias("unit_price")
)

order_items_clean.show()

+--------+-----------------+-------+-------------------+--------+----------+
|order_id|      customerRef|    sku|       product_name|quantity|unit_price|
+--------+-----------------+-------+-------------------+--------+----------+
|ord-9001|Customer/cust-001|SKU-100|     Wireless Mouse|       1|     24.99|
|ord-9001|Customer/cust-001|SKU-204|        USB-C Cable|       2|      9.99|
|ord-9002|Customer/cust-002|SKU-315|Mechanical Keyboard|       1|      89.5|
|ord-9003|Customer/cust-001|SKU-100|     Wireless Mouse|       1|     24.99|
|ord-9004|Customer/cust-003|SKU-500|      Monitor Stand|       1|      34.0|
|ord-9004|Customer/cust-003|SKU-204|        USB-C Cable|       3|      9.99|
|ord-9005|Customer/cust-002|SKU-315|Mechanical Keyboard|       1|      89.5|
|ord-9005|Customer/cust-002|SKU-100|     Wireless Mouse|       1|     24.99|
+--------+-----------------+-------+-------------------+--------+----------+



### Parse a Reference Field

`customerRef` looks like `"Customer/cust-001"` — same reference pattern as FHIR's `"Patient/abc-123"`. 

In [0]:
from pyspark.sql.functions import split

orders_clean = orders_raw.select(
    col("id").alias("order_id"),
    split(col("customerRef"), "/").getItem(1).alias("customer_id"),
    col("status"),
    col("orderDate").cast("timestamp").alias("order_date")
)

orders_clean.show()

+--------+-----------+---------+-------------------+
|order_id|customer_id|   status|         order_date|
+--------+-----------+---------+-------------------+
|ord-9001|   cust-001|completed|2026-02-10 14:22:00|
|ord-9002|   cust-002|  shipped|2026-03-01 09:05:00|
|ord-9003|   cust-001|completed|2026-03-18 17:40:00|
|ord-9004|   cust-003|cancelled|2026-04-05 11:12:00|
|ord-9005|   cust-002|completed|2026-04-22 13:30:00|
+--------+-----------+---------+-------------------+



### Join Across "Resource Types"

In [0]:
customer_orders = orders_clean.join(
    customers_clean,
    on="customer_id",
    how="left"
)

customer_orders.select(
    "order_id", "first_name", "last_name", "loyaltyTier", "status", "order_date"
).show()

+--------+----------+---------+-----------+---------+-------------------+
|order_id|first_name|last_name|loyaltyTier|   status|         order_date|
+--------+----------+---------+-----------+---------+-------------------+
|ord-9001|     Elena|   Rivera|       gold|completed|2026-02-10 14:22:00|
|ord-9002|    Marcus|     Chen|     silver|  shipped|2026-03-01 09:05:00|
|ord-9003|     Elena|   Rivera|       gold|completed|2026-03-18 17:40:00|
|ord-9004|     Priya|    Patel|       gold|cancelled|2026-04-05 11:12:00|
|ord-9005|    Marcus|     Chen|     silver|completed|2026-04-22 13:30:00|
+--------+----------+---------+-----------+---------+-------------------+



In [0]:
orphaned = orders_clean.join(
    customers_clean,
    on="customer_id",
    how="left_anti"
)

orphaned.count()  # should be 0 — every order should reference a real customer

0

### Compare `.explain()` on a Chained Transformation

In [0]:
result = (
    order_items_clean
    .join(orders_clean, on="order_id")
    .filter(col("status") == "completed")
    .select("order_id", "product_name", "quantity", "unit_price")
)

result.explain()
result.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [order_id#11903, product_name#11906, quantity#11907L, unit_price#11908]
         +- PhotonBroadcastHashJoin [order_id#11903], [order_id#11933], Inner, BuildRight, false, true
            :- PhotonProject [id#11702 AS order_id#11903, item#12038.productName AS product_name#11906, item#12038.quantity AS quantity#11907L, item#12038.unitPrice AS unit_price#11908]
            :  +- PhotonGenerate explode(items#11703), [id#11702], false, [item#12038]
            :     +- PhotonProject [id#11702, items#11703]
            :        +- PhotonFilter ((((isnotnull(resourceType#11707) AND isnotnull(id#11702)) AND (resourceType#11707 = Order)) AND isnotnull(items#11703)) AND (size(items#11703, false) > 0))
            :           +- PhotonJsonScan json [id#11702,items#11703,resourceType#11707] Batched: true, DataFilters: [isnotnull(resourceType#11707), is

### Write as Delta Tables

In [0]:
customers_clean.write.mode("overwrite").saveAsTable("learning.practice.customers")
orders_clean.write.mode("overwrite").saveAsTable("learning.practice.orders")
order_items_clean.write.mode("overwrite").saveAsTable("learning.practice.order_items")
contacts_clean.write.mode("overwrite").saveAsTable("learning.practice.contacts")

In [0]:
%sql
SELECT c.first_name, c.last_name, o.status, oi.product_name, oi.quantity
FROM learning.practice.orders o
JOIN learning.practice.customers c ON o.customer_id = c.customer_id
JOIN learning.practice.order_items oi ON o.order_id = oi.order_id
ORDER BY c.last_name

first_name,last_name,status,product_name,quantity
Marcus,Chen,completed,Mechanical Keyboard,1
Marcus,Chen,completed,Wireless Mouse,1
Marcus,Chen,shipped,Mechanical Keyboard,1
Priya,Patel,cancelled,Monitor Stand,1
Priya,Patel,cancelled,USB-C Cable,3
Elena,Rivera,completed,Wireless Mouse,1
Elena,Rivera,completed,USB-C Cable,2
Elena,Rivera,completed,Wireless Mouse,1
